<a href="https://colab.research.google.com/github/AnushaFreshStart/DL-agriculture-A/blob/main/Q9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Question 9

Task 1: Define the dataset directory, dataloader and model hyperparameters. The dataloader and model hyperparameters should be same as used during training.

In [ ]:
import os
import torch

# Define dataset directory
extract_dir = "."
dataset_path = os.path.join(extract_dir, "images_dataSAT")

# DataLoader & Image Hyperparameters
img_size = 64  # Spatial dimensions (64x64 pixels)
img_w, img_h = 64, 64
in_channels = 3  # RGB color channels
batch_size = 128  # Batch size matching training setup
num_classes = 2  # Binary land classification (non-agri vs agri)
lr = 0.001

# Vision Transformer / Hybrid Model Hyperparameters
patch_size = 16  # Patch dimension (16x16 pixels)
embed_dim = 256  # Token embedding dimension
depth = 6  # Number of Transformer encoder blocks
num_heads = 8  # Multi-head self-attention heads
mlp_dim = 2048  # Feed-forward hidden dimension
drop_rate = 0.1  # Dropout regularization rate

agri_class_labels = ["non-agri", "agri"]
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dataset Path: {dataset_path} | Running evaluation on: {device}")

Dataset Path: ./images_dataSAT | Running evaluation on: cuda


Task 2: Instantiate PyTorch model.

In [ ]:
# Instantiate the PyTorch Vision Transformer / CNN-ViT Hybrid model
pytorch_vit_model = VisionTransformer(
    img_size=img_size,
    patch_size=patch_size,
    in_channels=in_channels,
    num_classes=num_classes,
    embed_dim=embed_dim,
    depth=depth,
    num_heads=num_heads,
    mlp_dim=mlp_dim,
    drop_rate=drop_rate,
).to(device)

# Load saved best weights checkpoint if available
# pytorch_vit_model.load_state_dict(torch.load('pytorch_cnn_vit.pth'))
pytorch_vit_model.eval()
print("PyTorch CNN-ViT Hybrid Model successfully instantiated.")

PyTorch CNN-ViT Hybrid Model successfully instantiated.


Task 3: Print the evaluation metrics using print_metrics function for the Keras ViT model with name Keras CNN-Vit Hybrid Model.

In [ ]:
# Collect validation predictions from the trained Keras CNN-ViT hybrid
import numpy as np

val_gen.reset()
vit_probabilities = hybrid_model.predict(
    val_gen,
    steps=len(val_gen),
    verbose=0,
)
positive_class_index = val_gen.class_indices["class_1_agri"]

all_labels_keras_vit = np.asarray(val_gen.classes, dtype=int)
all_preds_keras_vit = np.argmax(vit_probabilities, axis=1)
all_probs_keras_vit = vit_probabilities[:, positive_class_index]

# Print evaluation metrics for Keras CNN-ViT Hybrid Model

print_metrics(
    all_labels_keras_vit,
    all_preds_keras_vit,
    all_probs_keras_vit,
    agri_class_labels,
    "Keras CNN-Vit Hybrid Model",
)

=== Keras CNN-Vit Hybrid Model Evaluation Metrics ===
Accuracy:  0.9875
Precision: 0.9867
Recall:    0.9883
F1-Score:  0.9875
ROC AUC:   0.9980



Task 4: Print the evaluation metrics using print_metrics function for the PyTorch ViT model with model name PyTorch CNN-Vit Hybrid Model.

In [ ]:
# Collect validation predictions from the PyTorch CNN-ViT hybrid
import numpy as np
import torch

pytorch_vit_model.eval()
all_labels_pytorch_vit = []
all_preds_pytorch_vit = []
all_probs_pytorch_vit = []
agri_index = agri_class_labels.index("agri")

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        logits = pytorch_vit_model(images)
        probabilities = torch.softmax(logits, dim=1)

        all_labels_pytorch_vit.extend(labels.cpu().numpy())
        all_preds_pytorch_vit.extend(logits.argmax(dim=1).cpu().numpy())
        all_probs_pytorch_vit.extend(probabilities[:, agri_index].cpu().numpy())

all_labels_pytorch_vit = np.asarray(all_labels_pytorch_vit, dtype=int)
all_preds_pytorch_vit = np.asarray(all_preds_pytorch_vit, dtype=int)
all_probs_pytorch_vit = np.asarray(all_probs_pytorch_vit)

# Print evaluation metrics for PyTorch CNN-ViT Hybrid Model
print_metrics(
    all_labels_pytorch_vit,
    all_preds_pytorch_vit,
    all_probs_pytorch_vit,
    agri_class_labels,
    "PyTorch CNN-Vit Hybrid Model",
)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


=== PyTorch CNN-Vit Hybrid Model Evaluation Metrics ===
Accuracy:  0.6492
Precision: 0.6933
Recall:    0.4826
F1-Score:  0.5691
ROC AUC:   0.6798

